In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv("diabetic_data.csv").replace("?", np.nan)

expired_hospice_ids = [11, 13, 14, 19, 20, 21]
df_clean = df[~df["discharge_disposition_id"].isin(expired_hospice_ids)].copy()

y = (df_clean["readmitted"] == "<30").astype(int)

low_miss_cols = ["race", "diag_3", "diag_2", "diag_1"]
low_miss_tbl = pd.DataFrame({
    "missing_rows": df_clean[low_miss_cols].isna().sum(),
    "missing_pct": (df_clean[low_miss_cols].isna().mean() * 100).round(2),
    "unique_vals": df_clean[low_miss_cols].nunique(dropna=True),
})
print(low_miss_tbl.to_string())

for col in low_miss_cols:
    df_clean[col] = df_clean[col].fillna("Missing")

med_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]
med_check_cols = med_cols + ["change", "diabetesMed"]

X_med_ord = OrdinalEncoder().fit_transform(df_clean[med_check_cols].astype(str))
mi_scores = mutual_info_classif(X_med_ord, y, discrete_features=True, random_state=42)

active_bin = pd.DataFrame({c: (df_clean[c] != "No").astype(int) for c in med_check_cols})
chi2_vals, chi2_pvals = chi2(active_bin, y)

med_audit = []
for idx, c in enumerate(med_check_cols):
    act_cnt = int((df_clean[c] != "No").sum())
    med_audit.append({
        "col": c,
        "active_rows": act_cnt,
        "active_pct": round(act_cnt / len(df_clean) * 100, 2),
        "chi2": round(chi2_vals[idx], 2),
        "chi2_p": round(chi2_pvals[idx], 4),
        "mutual_info": round(mi_scores[idx], 6),
    })

med_audit_df = pd.DataFrame(med_audit).sort_values(["active_rows", "chi2"], ascending=[True, True])
print(med_audit_df.to_string(index=False))

cols_to_drop = [
    "weight", "examide", "citoglipton", "acetohexamide",
    "glimepiride-pioglitazone", "metformin-pioglitazone",
    "metformin-rosiglitazone", "troglitazone", "glipizide-metformin",
    "tolbutamide", "miglitol", "tolazamide", "chlorpropamide",
    "acarbose", "nateglinide", "glyburide-metformin",
]
df_pruned = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns]).copy()


def map_icd9_chapter(val):
    if pd.isna(val) or str(val) in ["Missing", "nan", "None", "?"]:
        return "Missing"
    s = str(val).strip()
    if s.startswith("V"):
        return "V_Supplementary"
    if s.startswith("E"):
        return "E_External_Injury"
    try:
        num = float(s.split(".")[0])
    except ValueError:
        return "Other"
    if num == 250:
        return "Diabetes"
    if (390 <= num <= 459) or num == 785:
        return "Circulatory"
    if (460 <= num <= 519) or num == 786:
        return "Respiratory"
    if (520 <= num <= 579) or num == 787:
        return "Digestive"
    if 800 <= num <= 999:
        return "Injury_Poisoning"
    if 710 <= num <= 739:
        return "Musculoskeletal"
    if (580 <= num <= 629) or num == 788:
        return "Genitourinary"
    if 140 <= num <= 239:
        return "Neoplasms"
    return "Other"


df_ml = df_pruned.copy()
for dcol in ["diag_1", "diag_2", "diag_3"]:
    if dcol in df_ml.columns:
        df_ml[dcol] = df_ml[dcol].apply(map_icd9_chapter)

df_ml["target_lt30"] = (df_ml["readmitted"] == "<30").astype(int)
df_ml = df_ml.drop(columns=["readmitted"])

df_ml.to_csv("Final_diabetic_data.csv", index=False)
print(df_ml.shape)

        missing_rows  missing_pct  unique_vals
race            2234         2.25            5
diag_3          1419         1.43          786
diag_2           356         0.36          747
diag_1            20         0.02          715
                     col  active_rows  active_pct  chi2  chi2_p  mutual_info
                 examide            0        0.00   NaN     NaN     0.000000
             citoglipton            0        0.00   NaN     NaN     0.000000
           acetohexamide            1        0.00  0.13  0.7200     0.000001
glimepiride-pioglitazone            1        0.00  0.13  0.7200     0.000001
  metformin-pioglitazone            1        0.00  0.13  0.7200     0.000001
 metformin-rosiglitazone            2        0.00  0.26  0.6122     0.000002
            troglitazone            3        0.00  0.39  0.5346     0.000004
     glipizide-metformin           13        0.01  0.18  0.6748     0.000001
             tolbutamide           21        0.02  0.91  0.3391     0.00